In [1]:
import numpy as np
import random

class RecyclingRobotEnv:
    def __init__(self):
        self.states = [0, 1]
        self.state_labels = {0: "High", 1: "Low"}
        self.actions = [0, 1, 2]
        self.action_labels = {0: "Search", 1: "Wait", 2: "Recharge"}
        self.alpha = 0.8
        self.beta = 0.2
        self.r_search = 2.0
        self.r_wait = 0.5
        self.reset()

    def reset(self):
        self.state = 0
        return self.state

    def step(self, action):
        reward = 0
        done = False

        if self.state == 0:
            if action == 0:
                reward = self.r_search
                self.state = 0 if random.random() < self.alpha else 1
            elif action == 1:
                reward = self.r_wait
                self.state = 0
            elif action == 2:
                reward = 0
                self.state = 0

        elif self.state == 1:
            if action == 0:
                if random.random() < self.beta:
                    reward = self.r_search
                    self.state = 1
                else:
                    reward = -3.0
                    self.state = 0
            elif action == 1:
                reward = self.r_wait
                self.state = 1
            elif action == 2:
                reward = 0
                self.state = 0

        return self.state, reward, done


class QLearningAgent:
    def __init__(self, num_states, num_actions, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.q_table = np.zeros((num_states, num_actions))
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 1)
        return np.argmax(self.q_table[state])

    def learn(self, state, action, reward, next_state):
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state][best_next_action]
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error


if __name__ == "__main__":
    env = RecyclingRobotEnv()
    agent = QLearningAgent(num_states=2, num_actions=3, alpha=0.1, gamma=0.9, epsilon=0.15)

    state = env.reset()
    for _ in range(10000):
        action = agent.choose_action(state)
        next_state, reward, _ = env.step(action)
        agent.learn(state, action, reward, next_state)
        state = next_state

    print("--- Final Q-Table ---")
    print("State / Action | Search  | Wait    | Recharge")
    for s in env.states:
        print(f"{env.state_labels[s]:<14} | {agent.q_table[s][0]:.2f}    | {agent.q_table[s][1]:.2f}    | {agent.q_table[s][2]:.2f}")

    print("\n--- Optimal Policy Derived ---")
    for s in env.states:
        best_action = np.argmax(agent.q_table[s])
        print(f"When battery is {env.state_labels[s].upper()}, the robot should: {env.action_labels[best_action].upper()}")


--- Final Q-Table ---
State / Action | Search  | Wait    | Recharge
High           | 10.11    | 9.86    | 0.00
Low            | 6.78    | 6.79    | 0.00

--- Optimal Policy Derived ---
When battery is HIGH, the robot should: SEARCH
When battery is LOW, the robot should: WAIT
